The goal of this notebook is to test a basic RAG system on SEC 10-K data.

In [ ]:
%pip install numpy pdfplumber torch faiss-cpu transformers sentence-transformers tqdm flash-attn

This notebook is for testing Retrieval-Augmented Generation (RAG) on SEC 10-K data.

In [ ]:
import os
import json
import faiss
import torch
import pdfplumber
import numpy as np
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline
from sentence_transformers import SentenceTransformer
from tqdm import tqdm
from pathlib import Path

In [ ]:
# Confirm we are using a suitable runtime
!nvidia-smi --query-gpu=memory.total --format=csv,noheader,nounits

import torch

def check_gpu_memory():
    if torch.cuda.is_available():
        gpu_memory = torch.cuda.get_device_properties(0).total_memory / 1e9
        print(f"GPU Memory: {gpu_memory:.2f} GB")
        if gpu_memory >= 40:
            print("Sufficient VRAM: At least 40 GB available")
        else:
            print("Insufficient VRAM: Less than 40 GB available")
    else:
        print("No GPU available")

check_gpu_memory()

40960
GPU Memory: 42.47 GB
Sufficient VRAM: At least 40 GB available


In [ ]:
# Load the model, tokenizer, and dataset
from google.colab import drive
drive.mount('/content/drive')

import os

# Set up the cache directory
cache_dir = "/content/drive/My Drive/huggingface_cache"
os.makedirs(cache_dir, exist_ok=True)

# Model and device setup
device = 'cuda' if torch.cuda.is_available() else 'cpu'
retriever_model = SentenceTransformer('sentence-transformers/all-mpnet-base-v2').to(device)
rags_model = AutoModelForCausalLM.from_pretrained('microsoft/Phi-3.5-mini-instruct',
                                                 cache_dir=cache_dir,
                                                 trust_remote_code=True).to(device)
tokenizer = AutoTokenizer.from_pretrained('microsoft/Phi-3.5-mini-instruct',
                                                 cache_dir=cache_dir,
                                                 trust_remote_code=True)
generator = pipeline('text-generation', model=rags_model, tokenizer=tokenizer, device=0 if device=='cuda' else -1)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

Device set to use cuda:0


In [ ]:
# FAISS index setup
index_path = 'faiss_index.bin'
corpus_path = 'corpus.json'

# Global variables
index = None
corpus = []

# Load FAISS index and corpus if they exist
if os.path.exists(index_path):
    index = faiss.read_index(index_path)
else:
    index = faiss.IndexFlatL2(784)  # Initialize an empty FAISS index

if os.path.exists(corpus_path):
    with open(corpus_path, 'r') as f:
        corpus = json.load(f)

In [ ]:
# Load and preprocess PDFs
def extract_text_from_pdf(pdf_path):
    with pdfplumber.open(pdf_path) as pdf:
        text = '\n'.join(page.extract_text() or '' for page in pdf.pages)
    return text

# Chunking function
def chunk_text(text, chunk_size=500):
    words = text.split()
    return [' '.join(words[i:i+chunk_size]) for i in range(0, len(words), chunk_size)]

def chunk_text_semantic(text, max_chunk_size=500, overlap=50):
    """Chunk text with semantic awareness and overlap."""
    # Split text into paragraphs first
    paragraphs = [p for p in text.split('\n\n') if p.strip()]

    chunks = []
    current_chunk = []
    current_size = 0

    for para in paragraphs:
        para_words = para.split()
        para_size = len(para_words)

        # If adding this paragraph exceeds max size and we already have content
        if current_size + para_size > max_chunk_size and current_chunk:
            # Join the current chunk and add to chunks
            chunks.append(' '.join(current_chunk))
            # Keep some overlap with previous chunk
            overlap_words = current_chunk[-overlap:] if overlap < len(current_chunk) else current_chunk
            current_chunk = overlap_words + para_words
            current_size = len(current_chunk)
        else:
            # Add paragraph to current chunk
            current_chunk.extend(para_words)
            current_size += para_size

        # If current chunk exceeds max size, split it
        while current_size > max_chunk_size:
            chunks.append(' '.join(current_chunk[:max_chunk_size]))
            current_chunk = current_chunk[max_chunk_size-overlap:] if overlap < max_chunk_size else current_chunk[max_chunk_size:]
            current_size = len(current_chunk)

    # Add the last chunk if it has content
    if current_chunk:
        chunks.append(' '.join(current_chunk))

    return chunks

# Build FAISS index
def build_index(data_folder):
    global index, corpus
    pdf_files = list(Path(data_folder).rglob('*.pdf'))
    for pdf_file in tqdm(pdf_files, desc='Processing PDFs'):
        text = extract_text_from_pdf(pdf_file)
        chunks = chunk_text(text)
        # chunks = chunk_text_semantic(text)
        corpus.extend(chunks)
        embeddings = retriever_model.encode(chunks, convert_to_numpy=True)
        embedding_dim = embeddings.shape[1]
        if index.d != embedding_dim:
            index = faiss.IndexFlatL2(embedding_dim)
        index.add(embeddings)
    faiss.write_index(index, 'faiss_index.bin')
    with open('corpus.json', 'w') as f:
        json.dump(corpus, f)

# Retrieval function
def retrieve_relevant_chunks(query, top_k=5):
    query_embedding = retriever_model.encode([query], convert_to_numpy=True)
    distances, indices = index.search(query_embedding, top_k)

    # Check if the corpus is empty or indices are invalid before accessing it
    if not corpus or not indices.size or any(i >= len(corpus) for i in indices[0]):
        print("Warning: Corpus is empty or indices are out of range. Returning empty context.")
        return ["No context found for the query."] # Return a placeholder instead of an empty list
    return [corpus[i] for i in indices[0]]

# RAG generation function
def generate_response(query):
    query = query.replace("EGNIVIA", "NVIDIA") # Replace to pull the correct chunks
    print("Edited query to be:", query)
    relevant_chunks = retrieve_relevant_chunks(query)
    query = query.replace("NVIDIA", "EGNIVIA") # Replace prior to inference
    print("Repaired query to be:", query)
    context = '\n'.join(relevant_chunks)
    context = context.replace("NVIDIA", "EGNIVIA") # Replace context to avoid inteference from existing training data
    system_prompt = (
        "You are a financial analyst tasked with answering questions based ONLY on the provided context. "
        "If the answer cannot be found in the context, acknowledge this limitation and DO NOT provide information from your training. "
        "For numerical questions, cite specific numbers from the context. "
        "For analytical questions, base your reasoning explicitly on information in the context. "
        "Always indicate uncertainty when appropriate."
    )
    input_prompt = tokenizer.apply_chat_template([{"role": "system", "content": system_prompt}, {"role": "user", "content": f"Context:\n{context}\n\nQuery:\n{query}"}], tokenize=False, add_generation_prompt=True, return_tensors="pt")
    response = generator(input_prompt, max_new_tokens=256, do_sample=True)
    return response[0]['generated_text'][len(input_prompt):]


In [ ]:
build_index('/content/drive/My Drive/datasets/temp')

Processing PDFs: 100%|██████████| 1/1 [00:18<00:00, 18.63s/it]


In [ ]:
#if __name__ == '__main__':
    # import argparse
    # parser = argparse.ArgumentParser()
    # parser.add_argument('--data_folder', type=str, required=True)
    # parser.add_argument('--query', type=str, required=False, default=None)
    # args = parser.parse_args()

    # if args.query:
    #     print(generate_response(args.query))
    # else:
    #     build_index(args.data_folder)


In [ ]:
generate_response("What is EGNIVIA's 2023 revenue?")

Edited query to be: What is NVIDIA's 2023 revenue?
Repaired query to be: What is EGNIVIA's 2023 revenue?


OutOfMemoryError: CUDA out of memory. Tried to allocate 4.16 GiB. GPU 0 has a total capacity of 39.56 GiB of which 3.44 GiB is free. Process 7238 has 36.10 GiB memory in use. Of the allocated memory 34.24 GiB is allocated by PyTorch, and 1.38 GiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)

In [ ]:
generate_response("What is NVIDIA's 2023 target market?")

" The provided context does not specify a particular target market for NVIDIA in 2023. However, it discusses various market highlights and achievements across several areas such as Data Center, Gaming, Professional Visualization, and Automotive. Here's a brief summary:\n\n1. Data Center: Announced a $47.5 billion revenue, up 217%, and launched AI inference platforms optimized for AI workloads.\n2. Gaming: Launched RTX 4060 and 4070 GPUs, with significant growth (15%) in revenue, and announced the GeForce RTX 40-Series SUPER GPUs.\n3. Professional Visualization: Announced new GPUs based on the NVIDIA RTX Ada Lovelace architecture, and introduced NVIDIA Omniverse Cloud for industrial metaverse applications.\n4. Automotive: Announced a partnership with MediaTek to develop mainstream automotive systems on chips and furthered collaboration with Foxconn for next-generation electric vehicles.\n\nThese areas indicate NVIDIA's focus on leveraging its GPU architecture for A"

In [ ]:
generate_response("What are the primary differences between EGNIVIA's 2022 and 2023 SEC 10-K filings?")

'<|system|>\n9/16/2016 4.5 Form of 2026 Note 8-K 0-23985 Annex B-1 to 9/16/2016 Exhibit 4.2 4.6* Description of Securities 4.7 Officers’ Certificate, dated as of March 31, 2020 8-K 0-23985 4.2 3/31/2020 4.8 Form of 2030 Note 8-K 0-23985 Annex A-1 to 3/31/2020 Exhibit 4.2 4.9 Form of 2040 Note 8-K 0-23985 Annex B-1 to 3/31/2020 Exhibit 4.2 4.10 Form of 2050 Note 8-K 0-23985 Annex C-1 to 3/31/2020 Exhibit 4.2 4.11 Form of 2060 Note 8-K 0-23985 Annex D-1 to 3/31/2020 Exhibit 4.2 4.12 Officers\' Certificate, dated as of June 16, 2021 8-K 0-23985 4.2 6/16/2021 4.13 Form of 2023 Note 8-K 0-23985 Annex A-1 to 6/16/2021 Exhibit 4.2 4.14 Form of 2024 Note 8-K 0-23985 Annex B-1 to 6/16/2021 Exhibit 4.2 4.15 Form of 2028 Note 8-K 0-23985 Annex C-1 to 6/16/2021 Exhibit 4.2 4.16 Form of 2031 Note 8-K 0-23985 Annex D-1 to 6/16/2021 Exhibit 4.2 10.1 Form of Indemnity Agreement between NVIDIA Corporation 8-K 0-23985 10.1 3/7/2006 and each of its directors and officers 10.2+* Amended and Restated 2007 